# Day 16 – Evaluating LLMs – Extensive Solutions

Complete solutions for BLEU, ROUGE, BERTScore, and LLM‑as‑a‑judge.

In [ ]:
# !pip install evaluate nltk rouge-score bert-score
from evaluate import load
import nltk
nltk.download('punkt')

bleu = load("bleu")
rouge = load("rouge")
bertscore = load("bertscore")

print("Metrics loaded.")

## Exercise 1: Compare metrics on generated summaries

We'll generate 10 summaries using different models/temperatures and compute BLEU, ROUGE, BERTScore, and LLM judge.

In [ ]:
import openai
from dotenv import load_dotenv
import os
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def ask_gpt(prompt, temp=0.7):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    return response.choices[0].message.content

reference = "The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was built in 1889."
candidates = []
for temp in [0.2, 0.7, 1.2]:
    cand = ask_gpt(f"Summarize in one sentence: {reference}", temp=temp)
    candidates.append(cand)

# Compute metrics
for i, cand in enumerate(candidates):
    bleu_score = bleu.compute(predictions=[cand], references=[[reference]])
    rouge_score = rouge.compute(predictions=[cand], references=[reference])
    bert = bertscore.compute(predictions=[cand], references=[reference], lang="en")
    print(f"\nCandidate {i+1} (temp={[0.2,0.7,1.2][i]}): {cand[:80]}...")
    print(f"BLEU: {bleu_score['bleu']:.3f}")
    print(f"ROUGE-L: {rouge_score['rougeL']:.3f}")
    print(f"BERTScore F1: {bert['f1'][0]:.3f}")

## Exercise 2: Pairwise comparison with LLM judge

Ask GPT to choose which of two summaries is better.

In [ ]:
def pairwise_judge(reference, summary_a, summary_b):
    prompt = f"""Which summary is better (more accurate and complete)? Answer 'A' or 'B'.
    Reference: {reference}
    Summary A: {summary_a}
    Summary B: {summary_b}
    """
    response = ask_gpt(prompt, temp=0)
    return response.strip()

winner = pairwise_judge(reference, candidates[0], candidates[1])
print(f"Winner: {winner}")

## Exercise 3: Evaluate your RAG system

Create a test set of 20 questions with golden answers, compute BERTScore for each.

In [ ]:
# Example test set
test_questions = [
    ("What is the capital of France?", "Paris"),
    ("How tall is the Eiffel Tower?", "330 meters"),
]

# Simulate RAG answers (replace with your actual RAG call)
def rag_answer(question):
    # In reality, call your RAG chain
    return ask_gpt(f"Answer concisely: {question}", temp=0)

scores = []
for q, gold in test_questions:
    pred = rag_answer(q)
    bert = bertscore.compute(predictions=[pred], references=[gold], lang="en")
    scores.append(bert['f1'][0])
    print(f"Q: {q}\nPred: {pred}\nGold: {gold}\nBERTScore: {bert['f1'][0]:.3f}\n")

print(f"Average BERTScore: {sum(scores)/len(scores):.3f}")

## Exercise 4: Hallucination metric

Design a simple metric that penalises facts not in the source. We'll use keyword overlap or NLI.

In [ ]:
def hallucination_score(source, answer):
    # Simple: count of novel entities (words not in source)
    source_words = set(source.lower().split())
    answer_words = set(answer.lower().split())
    novel = answer_words - source_words
    return len(novel) / max(len(answer_words), 1)

source = "The Eiffel Tower is in Paris."
good_answer = "The Eiffel Tower is located in Paris."
hallucinated = "The Eiffel Tower is in London."
print(f"Good answer hallucination score: {hallucination_score(source, good_answer):.2f}")
print(f"Hallucinated answer score: {hallucination_score(source, hallucinated):.2f}")